# 99 · Recovery — clean up broken live ingestion (May 11–15)

**One-time recovery notebook.** Run the cells **in order**, stopping at each verification cell to inspect output before continuing.

## Why we are here
`00_live_pull.ipynb` was pulling yfinance without flattening MultiIndex columns. The Spark conversion silently nulled every OHLCV value (AAPL escaped because it was ingested via another path earlier). Result: 5 broken `live_*.parquet` files in `bronze/uploads/`, each ingested 2–6 times into `bronze/prices/`, and corrupted Silver/Gold for Date >= 2026-05-11.

## Plan
1. **Diagnose** — read-only inspection of uploads + bronze partitions + silver state.
2. **Cleanup** — delete broken upload files and broken bronze partitions.
3. **Backfill** — re-pull May 11–15 with the MultiIndex fix; write **one clean file** to a dedicated subfolder (not the main uploads/, to avoid re-ingesting any leftover seed/batch_2 files).
4. **Bronze ingest** — run `01_bronze_ingest` pointed at the backfill subfolder.
5. **Silver fix** — `DELETE FROM silver WHERE Date >= '2026-05-11'` (Silver is Delta).
6. **Silver + Gold rebuild** — run `02_silver_build` and `03_gold_build`.
7. **Verify** — all 26 tickers show Date = 2026-05-15 with non-null close in `gold.dashboard_latest_prices`.

## Key design choice
We stage the backfill file under `bronze/backfill/` instead of `bronze/uploads/`. `01_bronze_ingest` uses `recursiveFileLookup=true` over its source folder — if old seed/batch_2 parquet files are still sitting in `bronze/uploads/`, they would be re-ingested as duplicates. Pointing the widget at a dedicated folder eliminates that risk.

In [ ]:
from datetime import date, timedelta
import pandas as pd
import yfinance as yf
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DateType, DoubleType, LongType,
)

UPLOADS_PATH  = "/Volumes/asset_mgmt/bronze/uploads/"
BACKFILL_PATH = "/Volumes/asset_mgmt/bronze/backfill/"   # dedicated staging for this recovery
PRICES_PATH   = "/Volumes/asset_mgmt/bronze/prices/"
SILVER_TABLE  = "asset_mgmt.silver.daily_prices"
GOLD_TABLE    = "asset_mgmt.gold.dashboard_latest_prices"

# yfinance end is exclusive; this captures Mon 5/11 → Fri 5/15
BACKFILL_START = "2026-05-11"
BACKFILL_END   = "2026-05-16"

TICKERS = [
    "AAPL", "MSFT", "NVDA", "GOOGL", "META",
    "JPM",  "BAC",  "GS",   "MS",    "WFC",
    "JNJ",  "UNH",  "PFE",  "ABBV",  "MRK",
    "XOM",  "CVX",  "COP",  "SLB",   "EOG",
    "AMZN", "HD",   "MCD",  "NKE",   "COST",
    "SPY",
]

dbutils.fs.mkdirs(BACKFILL_PATH)
print(f"backfill staging : {BACKFILL_PATH}")

## Phase 1 · Diagnostics (read-only)

Inspect the three outputs below before proceeding. Confirm:
- which files sit in `uploads/`
- which `ingest_date` partitions in `bronze/prices/` contain broken live data (null Close)
- the current Silver state for Date >= 2026-05-11

In [ ]:
# 1a) what is in uploads/
print("--- uploads/ ---")
for f in sorted(dbutils.fs.ls(UPLOADS_PATH), key=lambda x: x.name):
    kind = "dir " if f.isDir() else "file"
    print(f"  {kind}  {f.size:>12,}  {f.name}")

In [ ]:
# 1b) bronze partitions — rows, distinct files, and how many have null Close
bronze = spark.read.parquet(PRICES_PATH)

display(
    bronze
    .groupBy("ingest_date", "_source_type")
    .agg(
        F.count("*").alias("rows"),
        F.countDistinct("Ticker").alias("tickers"),
        F.countDistinct("_source_file").alias("distinct_files"),
        F.sum(F.when(F.col("Close").isNull(), 1).otherwise(0)).alias("null_close_rows"),
        F.min("Date").alias("date_min"),
        F.max("Date").alias("date_max"),
    )
    .orderBy("ingest_date", "_source_type")
)

In [ ]:
# 1c) silver state for the affected window
display(
    spark.table(SILVER_TABLE)
    .filter(F.col("Date") >= "2026-05-08")
    .groupBy("Date")
    .agg(
        F.countDistinct("Ticker").alias("tickers"),
        F.count("*").alias("rows"),
    )
    .orderBy("Date")
)

## Phase 2 · Cleanup (DESTRUCTIVE)

After reviewing the diagnostics above, run the cells below. They:
1. delete broken `live_*.parquet` files from `uploads/`
2. delete any prior failed backfill attempt
3. delete every `bronze/prices/ingest_date=*` partition that contains null-Close live rows

Seed and batch_2 partitions live under different `ingest_date` folders and are not touched.

In [ ]:
# 2a) compute the list of broken bronze partitions (the cell prints them; nothing destructive yet)
broken_partitions = sorted(
    r[0] for r in (
        bronze
        .filter((F.col("_source_type") == "live") & F.col("Close").isNull())
        .select("ingest_date").distinct()
        .collect()
    )
)

print(f"Broken bronze partitions (will be deleted): {len(broken_partitions)}")
for d in broken_partitions:
    print(f"  {PRICES_PATH}ingest_date={d}")

In [ ]:
# 2b) delete broken upload files (and any prior backfill attempt)
targets = [
    "live_2026-05-11.parquet",
    "live_2026-05-12.parquet",
    "live_2026-05-13.parquet",
    "live_2026-05-14.parquet",
    "live_2026-05-15.parquet",
    "live_backfill_may9_16.parquet",   # the earlier failed attempt
]
for name in targets:
    path = f"{UPLOADS_PATH}{name}"
    try:
        ok = dbutils.fs.rm(path, recurse=True)
        print(f"  {'rm  ' if ok else 'miss'}  {path}")
    except Exception as e:
        print(f"  err   {path}: {e}")

In [ ]:
# 2c) delete the broken bronze partitions
for d in broken_partitions:
    path = f"{PRICES_PATH}ingest_date={d}"
    try:
        ok = dbutils.fs.rm(path, recurse=True)
        print(f"  {'rm  ' if ok else 'miss'}  {path}")
    except Exception as e:
        print(f"  err   {path}: {e}")

In [ ]:
# 2d) verify bronze is clean — expect ZERO rows where _source_type='live' AND Close IS NULL
bronze_after = spark.read.parquet(PRICES_PATH)

display(
    bronze_after
    .groupBy("ingest_date", "_source_type")
    .agg(
        F.count("*").alias("rows"),
        F.sum(F.when(F.col("Close").isNull(), 1).otherwise(0)).alias("null_close_rows"),
    )
    .orderBy("ingest_date", "_source_type")
)

remaining_bad = (
    bronze_after
    .filter((F.col("_source_type") == "live") & F.col("Close").isNull())
    .count()
)
assert remaining_bad == 0, f"Still {remaining_bad} broken live rows in bronze — stop and investigate"
print("OK: bronze contains no null-Close live rows")

## Phase 3 · Backfill — pull May 11–15 with MultiIndex fix

Pulls each ticker individually, flattens MultiIndex columns, concatenates, asserts no null Close, then writes **one** Parquet file to `bronze/backfill/`.

In [ ]:
%pip install yfinance --quiet

In [ ]:
# 3a) pull and flatten
frames = []
failed = []

for ticker in TICKERS:
    try:
        df = yf.download(
            ticker,
            start=BACKFILL_START,
            end=BACKFILL_END,
            auto_adjust=True,
            progress=False,
        )
        if df.empty:
            print(f"  [SKIP] {ticker}: empty response")
            continue
        # The fix: flatten MultiIndex columns BEFORE selecting OHLCV
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        df = df[["Open", "High", "Low", "Close", "Volume"]].copy()
        df.index.name = "Date"
        df.reset_index(inplace=True)
        df.insert(0, "Ticker", ticker)
        frames.append(df)
    except Exception as exc:
        print(f"  [ERROR] {ticker}: {exc}")
        failed.append(ticker)

pandas_df = pd.concat(frames, ignore_index=True)
pandas_df["Date"] = pd.to_datetime(pandas_df["Date"]).dt.date

print(f"\nTickers with data : {len(frames)} / {len(TICKERS)}")
print(f"Failed            : {failed if failed else 'none'}")
print(f"Total rows        : {len(pandas_df):,}")
print(f"Date range        : {pandas_df['Date'].min()} → {pandas_df['Date'].max()}")
print(f"Null Close rows   : {int(pandas_df['Close'].isna().sum())}  (must be 0)")
print(f"Distinct tickers  : {pandas_df['Ticker'].nunique()}")

# Hard guards — stop here if anything is off
assert pandas_df['Close'].isna().sum() == 0, "Null Close in pull — MultiIndex fix did not take effect"
assert len(failed) == 0, f"yfinance failures: {failed}"
assert pandas_df['Ticker'].nunique() == 26, f"Expected 26 tickers, got {pandas_df['Ticker'].nunique()}"
assert not pandas_df.duplicated(subset=['Ticker', 'Date']).any(), "Duplicate (Ticker, Date) in pull"
print("\nOK: pull is clean, complete, and unique on (Ticker, Date)")

In [ ]:
# 3b) write ONE clean Parquet file to bronze/backfill/
# Wipe the backfill folder first so it contains exactly one file at the end.
for f in dbutils.fs.ls(BACKFILL_PATH):
    dbutils.fs.rm(f.path, recurse=True)
    print(f"  rm  {f.path}")

schema = StructType([
    StructField("Ticker", StringType(),  False),
    StructField("Date",   DateType(),    False),
    StructField("Open",   DoubleType(),  True),
    StructField("High",   DoubleType(),  True),
    StructField("Low",    DoubleType(),  True),
    StructField("Close",  DoubleType(),  True),
    StructField("Volume", LongType(),    True),
])

spark_df = (
    spark.createDataFrame(pandas_df, schema=schema)
        .withColumn("_source_type",      F.lit("live"))
        .withColumn("_ingest_timestamp", F.current_timestamp())
)

out_path = f"{BACKFILL_PATH}live_backfill_2026-05-11_to_2026-05-15.parquet"
(
    spark_df.coalesce(1)
        .write
        .mode("overwrite")
        .parquet(out_path)
)
print(f"\nWritten → {out_path}")

In [ ]:
# 3c) verify the backfill file actually landed on disk and read-back is clean
print("--- backfill/ ---")
for f in sorted(dbutils.fs.ls(BACKFILL_PATH), key=lambda x: x.name):
    kind = "dir " if f.isDir() else "file"
    print(f"  {kind}  {f.size:>12,}  {f.name}")

readback = spark.read.parquet(BACKFILL_PATH)
rb_rows  = readback.count()
rb_null  = readback.filter(F.col("Close").isNull()).count()
rb_tickr = readback.select("Ticker").distinct().count()

print(f"\nRead-back rows    : {rb_rows:,}")
print(f"Null Close rows   : {rb_null}")
print(f"Distinct tickers  : {rb_tickr}")

assert rb_null == 0,    "Backfill file has null Close rows"
assert rb_tickr == 26,  f"Backfill file has {rb_tickr} tickers, expected 26"
print("\nOK: backfill file is on disk, clean, and complete")

## Phase 4 · Bronze ingest — run `01_bronze_ingest`

Open **`01_bronze_ingest.ipynb`** and set the widgets to:

| widget        | value                                       |
|---------------|---------------------------------------------|
| `source_type` | `live`                                      |
| `uploads_path`| `/Volumes/asset_mgmt/bronze/backfill/`      |

Then run the whole notebook. It will read the one backfill file, stamp `_source_type=live` + `ingest_date=<today>`, and append to `bronze/prices/`.

**Come back here after it finishes** and run the verification cell below.

In [ ]:
# 4a) verify bronze now has clean live rows for May 11–15
bronze_v2 = spark.read.parquet(PRICES_PATH)

display(
    bronze_v2
    .filter(F.col("Date") >= "2026-05-11")
    .groupBy("Date", "_source_type")
    .agg(
        F.countDistinct("Ticker").alias("tickers"),
        F.count("*").alias("rows"),
        F.sum(F.when(F.col("Close").isNull(), 1).otherwise(0)).alias("null_close"),
    )
    .orderBy("Date", "_source_type")
)

# Each trading date should have exactly 26 tickers, exactly 26 rows, zero nulls
check = (
    bronze_v2
    .filter((F.col("_source_type") == "live") & (F.col("Date") >= "2026-05-11"))
    .groupBy("Date")
    .agg(F.count("*").alias("rows"), F.countDistinct("Ticker").alias("tickers"))
    .collect()
)
for row in check:
    assert row["rows"] == 26 and row["tickers"] == 26, f"Bad row for {row['Date']}: {row.asDict()}"
print("OK: bronze has 26 unique tickers per date for the backfill window")

## Phase 5 · Silver fix — delete affected rows

Silver is a Delta table, so we can DELETE in place. We drop everything from 2026-05-11 onward; `02_silver_build` will re-insert correct rows from the clean Bronze.

In [ ]:
spark.sql(f"""
DELETE FROM {SILVER_TABLE}
WHERE Date >= DATE '2026-05-11'
""")

display(
    spark.table(SILVER_TABLE)
    .filter(F.col("Date") >= "2026-05-08")
    .groupBy("Date")
    .agg(F.countDistinct("Ticker").alias("tickers"))
    .orderBy("Date")
)
print("Silver rows for Date >= 2026-05-11 deleted")

## Phase 6 · Rebuild Silver and Gold

Run these two notebooks in order (top to bottom, no widget changes needed):

1. **`02_silver_build.ipynb`**
2. **`03_gold_build.ipynb`**

Then return here and run the final verification cell.

## Phase 7 · Final verification

In [ ]:
# 7) every ticker should show latest_date = 2026-05-15 with a non-null close
gold = spark.table(GOLD_TABLE)

display(gold.orderBy("Sector", "Ticker"))

max_date = gold.agg(F.max("latest_date")).collect()[0][0]
stale    = gold.filter(F.col("latest_date") < "2026-05-15").collect()
null_close_count = gold.filter(F.col("close").isNull()).count()
row_count = gold.count()

print(f"Total rows         : {row_count}  (expect 26)")
print(f"Max latest_date    : {max_date}  (expect 2026-05-15)")
print(f"Null close rows    : {null_close_count}  (expect 0)")

if stale:
    print("\nSTALE TICKERS — latest_date < 2026-05-15:")
    for r in stale:
        print(f"  {r['Ticker']}: {r['latest_date']}")
else:
    print("\nOK: all 26 tickers show latest_date = 2026-05-15")

assert row_count == 26
assert null_close_count == 0
assert not stale, "Stale tickers remain — see list above"

## Done

After this notebook runs cleanly:

- `bronze/uploads/` no longer contains broken live files
- `bronze/backfill/` contains one clean Parquet file (you can delete the folder after the assignment if you want)
- `bronze/prices/` has one good live partition for the May 11–15 window
- `silver.daily_prices` and both Gold tables are correct for all 26 tickers through 2026-05-15

Tonight's scheduled job (`00_live_pull` at 22:30 Europe/Paris) will trigger but will exit at the weekend guard — today is Saturday, so there is no May 16 trading data to fetch. The first real production run with the MultiIndex fix in effect will be **Monday 2026-05-18 at 22:30 Europe/Paris**.

If you want, you can do a dry-run of `00_live_pull` *now* against a weekday range to convince yourself the fix is wired in — but the recovery above has already exercised the exact same `yf.download → flatten → select → createDataFrame` code path, so it is already proven.